In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone, timedelta

from lib.data import get_cached_data
from lib.analysis import (
    detect_candlestick_patterns,
    find_support_resistance,
    analyze_market_structure,
    build_price_action_signal,
)
from lib.ai import analyze_with_ai

In [ ]:
# Higher timeframe data (daily)
print("=" * 60)
print("📊 DAILY TIMEFRAME (1mo de données)")
print("=" * 60)
df_daily = get_cached_data("EURUSD=X", interval="1d", period="1mo")

# Higher timeframe data (1h - proxy pour 4h)
# Note: yfinance/OpenBB ne supporte pas '4h', on utilise '1h' à la place
print("\n" + "=" * 60)
print("📊 1H TIMEFRAME (1mo de données)")
print("=" * 60)
df_1h = get_cached_data("EURUSD=X", interval="1h", period="1mo")

In [ ]:
# Analyse daily
daily_levels = find_support_resistance(df_daily)
daily_structure = analyze_market_structure(df_daily)

print("📈 Daily - Structure de marché:")
print(f"   Trend: {daily_structure['structure']}")
print(f"   Bias: {daily_structure['bias']}")
print(f"   Range: {daily_structure['price_range_pct']:.2f}%")
print()

print("🎯 Daily - Niveaux clés:")
for lvl in daily_levels:
    emoji = "🟢" if lvl['type'] == 'Support' else "🔴"
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

# Analyse 1h
levels_1h = find_support_resistance(df_1h)
structure_1h = analyze_market_structure(df_1h)

print("\n" + "─" * 50)
print("📈 1H - Structure de marché:")
print(f"   Trend: {structure_1h['structure']}")
print(f"   Bias: {structure_1h['bias']}")
print(f"   Range: {structure_1h['price_range_pct']:.2f}%")
print()

print("🎯 1H - Niveaux clés:")
for lvl in levels_1h:
    emoji = "🟢" if lvl['type'] == 'Support' else "🔴"
    print(f"   {emoji} {lvl['type']}: {lvl['level']:.5f} ({lvl['strength']}, {lvl['touches']} touches)")

In [ ]:
# Données intraday (5m) pour le signal
print("📊 INTRADAY TIMEFRAME (5m)")
print("=" * 60)
df_5m = get_cached_data("EURUSD=X", interval="5m", period="1d")

# Patterns et signal intraday
patterns_5m = detect_candlestick_patterns(df_5m)
levels_5m = find_support_resistance(df_5m)
structure_5m = analyze_market_structure(df_5m)

signal = build_price_action_signal(df_5m, patterns_5m, levels_5m, structure_5m)

In [ ]:
# Enrichir le signal avec le contexte des plus hautes timeframes
signal['htf_daily_bias'] = daily_structure['bias']
signal['htf_daily_structure'] = daily_structure['structure']
signal['htf_1h_bias'] = structure_1h['bias']
signal['htf_1h_structure'] = structure_1h['structure']

# Ajouter les niveaux daily et 1h comme contexte
signal['daily_support'] = [l['level'] for l in daily_levels if l['type'] == 'Support'][:2]
signal['daily_resistance'] = [l['level'] for l in daily_levels if l['type'] == 'Resistance'][:2]
signal['hourly_support'] = [l['level'] for l in levels_1h if l['type'] == 'Support'][:2]
signal['hourly_resistance'] = [l['level'] for l in levels_1h if l['type'] == 'Resistance'][:2]

print("\n📊 SIGNAL COMBINÉ (Multi-Timeframe)")
print("=" * 70)
print(f"Daily bias:    {daily_structure['bias']:>8}  |  1H bias:      {structure_1h['bias']:>8}")
print(f"Intraday bias: {structure_5m['bias']:>8}  |  Intraday price: {signal['price']:.5f}")

# Vérifier la concordance des timeframes
biases = [daily_structure['bias'], structure_1h['bias'], structure_5m['bias']]
if all(b == 'Bullish' for b in biases):
    print("\n✅ TOUS LES TIMEFRAMES SONT BULLISH - Signal haussier fort")
elif all(b == 'Bearish' for b in biases):
    print("\n✅ TOUS LES TIMEFRAMES SONT BEARISH - Signal baissier fort")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bullish' and structure_5m['bias'] != 'Bullish':
    print("\n⚠️  HTF bullish mais intraday en contretendance - Attendre confirmation")
elif daily_structure['bias'] == structure_1h['bias'] == 'Bearish' and structure_5m['bias'] != 'Bearish':
    print("\n⚠️  HTF bearish mais intraday en contretendance - Attendre confirmation")
else:
    print("\n🔄 Timeframes en désaccord - Prudence recommandée")

In [ ]:
# Analyse IA avec le contexte multi-timeframe
print("🤖 ANALYSE IA MULTI-TIMEFRAME")
print("=" * 70)

ai_result = analyze_with_ai(signal)

if ai_result:
    signal_emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "🟡"}
    ai_signal = ai_result.get('signal', 'UNKNOWN').upper()
    confidence = ai_result.get('confidence', 0)

    print(f"\nSignal:          {ai_signal} {signal_emoji.get(ai_signal, '⚪')}")
    print(f"Confiance:       {confidence}%")
    print(f"Raison:          {ai_result.get('reason', 'N/A')}")

    if 'entry' in ai_result:
        print(f"Entrée:          {ai_result.get('entry', 'N/A')}")
        print(f"Stop Loss:       {ai_result.get('stop_loss', 'N/A')}")
        print(f"Take Profit:     {ai_result.get('take_profit', 'N/A')}")
else:
    print("Analyse IA indisponible")